# ⚙️ Lógica Forense del Optimizador: El Modelo de Muelles Virtuales

Este notebook documenta la arquitectura matemática del motor de optimización VRP utilizado en el proyecto, centrándose en el concepto de **Muelles Virtuales** para el balanceo de carga.

### 🏛️ Estructura del Nodo: Clonación Dinámica

Para gestionar flotas específicas por planta sin romper la linealidad del solver de OR-Tools, implementamos una técnica denominada "Muelles Virtuales". Cada planta física con $N$ camiones se descompone en $N$ nodos clonados (muelles):

$$ \mathcal{N}_{muelles}(P) = \{P_1, P_2, \dots, P_N\} $$

Cada muelle $P_i$ actúa como un puerto de salida dedicado para un único vehículo, garantizando que no se exceda la capacidad de flota de cada centro de distribución.

### ⚖️ Función de Penalización de Balanceo

El balanceo de rutas se logra mediante una función de coste que penaliza las desviaciones del promedio de carga o distancia. La función de penalización cuadrática (estilo muelle elástico) se define como:

$$ J = \sum_{i=1}^{V} d_i + k \cdot \sum_{n \in dropped} P_n + \text{Penalty}_{retorno} $$

Donde:
- $d_i$: Distancia de la ruta $i$.
- $k$: Constante de penalización por nodos no visitados.
- $\text{Penalty}_{retorno}$: Factor multiplicador para el arco de retorno al depósito ($d_{last} \times 2.5$) que fuerza la optimización de los tramos de "bajada".

In [5]:
import sys
import os
import pandas as pd
import numpy as np

# Añadir el directorio raíz al path para importar logistic_core
sys.path.append(os.path.abspath(os.path.join('..', '..', '..')))

from logistic_core.engine.solver import LogisticsSolver

def generate_tfm_mock_data(n_clientes=10, n_plantas=1):
    """Genera un diccionario de datos sintéticos compatible con LogisticsSolver."""
    data = {
        'paper_plant': {'id': 'DEPOT', 'name': 'Mengíbar Depot', 'lat': 38.0, 'lng': -3.8},
        'carton_plants': []
    }
    for i in range(n_plantas):
        plant_id = f"CP{i+1:03d}"
        plant = {
            'id': plant_id,
            'name': f"Planta de Cartón {i+1}",
            'lat': 38.0 + (i+1)*0.1,
            'lng': -3.8 + (i+1)*0.1,
            'customers': []
        }
        for j in range(n_clientes):
            plant['customers'].append({
                'id': f"CUST_{plant_id}_{j+1:03d}",
                'name': f"Cliente {j+1} de {plant_id}",
                'lat': plant['lat'] + (j+1)*0.01,
                'lng': plant['lng'] + (j+1)*0.01,
                'demanda_pallets': 5
            })
        data['carton_plants'].append(plant)
    return data

# Generamos datos sintéticos para la demostración de balanceo
mock_data = generate_tfm_mock_data(n_clientes=12, n_plantas=2)
solver = LogisticsSolver(data=mock_data)

print(f"Solver inicializado con {len(mock_data['carton_plants'])} plantas y {len(mock_data['carton_plants'][0]['customers'])} clientes por planta.")

2026-04-13 20:06:14,276 | INFO     | logistic_core.engine.solver | Nodos parseados: 27 (1 depósito, 2 plantas, 24 clientes)
2026-04-13 20:06:14,502 | INFO     | logistic_core.utils.geo | Matriz OSM recuperada íntegramente de la caché.


Solver inicializado con 2 plantas y 12 clientes por planta.


## 1. Visualización de Muelles Virtuales

Simulamos una flota específica de 3 camiones para la planta principal y observamos cómo se clonan los nodos internos para el solver.

In [4]:
flota_config = {mock_data['carton_plants'][0]['id']: 3}
print(f"Configuración de Flota: {flota_config}")

# Ejecutamos el solver (esto activa la clonación de nodos internamente)
# IMPORTANTE: Passamos max_pallets_ruta para evitar errores de dimensión en OR-Tools
routes = solver.solve(
    flota_por_planta=flota_config,
    max_pallets_ruta=33, 
    max_search_time=2
)

# Mostramos los clones generados en la estructura interna
for node in solver.nodes:
    if node['type'] == 'carton_plant' or '_clone_' in str(node.get('id', '')):
        print(f"Nodo Clonado: {node['id']} ({node['name']})")

2026-04-13 20:06:00,299 | INFO     | logistic_core.engine.solver | Nodos parseados para optimizar (con muelles virtuales): 29
2026-04-13 20:06:00,300 | INFO     | logistic_core.engine.solver | Modo: FLOTA ESPECÍFICA (Clonación de Nodos) | max_plantas_ruta=1 | n_clientes=12 | vehículos=4
2026-04-13 20:06:00,302 | INFO     | logistic_core.engine.solver | Iniciando optimización (4 vehículos, límite 2s, algoritmo: GUIDED_LOCAL_SEARCH)...
2026-04-13 20:06:00,304 | INFO     | logistic_core.engine.solver | Estadísticas de Matriz: Min(>0)=594.0m, Max=60396.1m


Configuración de Flota: {'CP001': 3}


2026-04-13 20:06:02,305 | INFO     | logistic_core.engine.solver | Solver terminó con éxito (status 1).
2026-04-13 20:06:02,307 | INFO     | logistic_core.engine.solver | Solución encontrada: 3 rutas activas.
2026-04-13 20:06:02,308 | INFO     | logistic_core.engine.drop_logger | DropLogger ha generado el reporte avanzado (sin emojis) en: logs\descartados_motivos.log


Nodo Clonado: CP001 (Planta de Cartón 1)
Nodo Clonado: CP002 (Planta de Cartón 2)


### Conclusión del Modelo
La arquitectura de **Muelles Virtuales** permite al optimizador de Google (OR-Tools) tratar las restricciones de flota local como restricciones de visita de nodos, lo cual es computacionalmente más eficiente y garantiza el cumplimiento estricto del número de vehículos disponibles por centro sin necesidad de modificar el corazón del algoritmo heurístico.